# Variational Autoencoders: ELBO, Inference, and Learning

The **variational autoencoder** is one of the clearest examples of how a deep generative model grows out of a probabilistic bottleneck. We begin with a latent-variable model $p_\theta(\boldsymbol{x}, \boldsymbol{z}) = p_\theta(\boldsymbol{x} | \boldsymbol{z}) p(\boldsymbol{z})$. This already gives a plausible generative story: first draw a latent cause $\boldsymbol{z}$ from a simple prior, then generate an observation $\boldsymbol{x}$ from the conditional distribution $p_\theta(\boldsymbol{x} | \boldsymbol{z})$.

The difficulty is not in telling that story, but in learning it from data. To evaluate or maximize the data likelihood, we would need the marginal
:::{math}
p_\theta(\boldsymbol{x}) = \int p_\theta(\boldsymbol{x} | \boldsymbol{z}) p(\boldsymbol{z}) \, d\boldsymbol{z},
:::
and that integral is usually intractable. At the same time, the posterior $p_\theta(\boldsymbol{z} | \boldsymbol{x})$ is exactly the object that would tell us which latent explanations are plausible for a given image, yet it is also typically unavailable in closed form.

So before introducing any architecture, it helps to name the real problem clearly: a VAE is a way of making **latent-variable learning** possible when both the marginal likelihood and the exact posterior are hard.


The next move is therefore methodological before it is architectural. We introduce a second distribution, the **approximate posterior** $q_\phi(\boldsymbol{z} | \boldsymbol{x})$, whose job is to imitate the true posterior well enough that learning becomes tractable. In practice, this approximate posterior is implemented by an encoder network, also called a recognition model. The word *variational* signals exactly this point: we are replacing exact inference with an optimization problem over a chosen family of approximate posterior distributions.

This chapter deliberately mixes **the probabilistic derivation**, **the architectural viewpoint**, and **the implementation viewpoint**. A VAE is not really understood if one only proves the ELBO and leaves the reader to guess later how the posterior family, the latent dimension, the observation model, and the KL weighting are encoded in the actual model. The aim here is to keep those pieces aligned from the beginning.


## Architecture and Modeling Choices

```{figure} ../assets/images/VAE_architecture.png
:width: 76%
:align: center

A convolutional variational autoencoder. The encoder maps an image to the parameters of a latent Gaussian, the stochastic bottleneck samples a code, and the decoder maps that code back into image logits.
```

The figure makes a key point visually: a VAE is not just “an autoencoder plus noise.” The left half is an **inference model** that approximates the posterior. The middle is a **stochastic bottleneck** that prevents the latent representation from being a deterministic lookup table. The right half is a **generative model** that explains how latent causes are turned back into pixels.

The architecture used in this chapter is intentionally modest but principled. We use convolutions because images have spatial locality. We use a relatively small latent dimension because the bottleneck should do real work. And we let the decoder output logits rather than probabilities because that is the numerically stable way to optimize a Bernoulli-style observation model.

This is the first chapter where all three strands of the course fully meet: neural parameterization, probabilistic latent-variable modeling, and tractable optimization. The previous chapter ended by identifying the two hard quantities in a deep latent-variable model, namely the marginal likelihood and the exact posterior. The VAE should therefore be read as the first serious answer to that bottleneck. It does not remove the intractability by magic. It reorganizes it into a lower bound that can be optimized with neural encoders and decoders.

A simple mental picture helps here. Imagine a dataset of shoes. A useful latent code might organize broad factors such as ankle boot versus sandal, heel height, or overall silhouette. The decoder then learns how those hidden factors map back into pixels. The encoder tries to reverse that map from a specific image. The VAE is the mathematical framework that makes this bidirectional story trainable.

## Derivation of the Evidence Lower Bound

The derivation begins with the identity
:::{math}
\log p_\theta(\boldsymbol{x})
=
\log \int q_\phi(\boldsymbol{z} | \boldsymbol{x})
\frac{p_\theta(\boldsymbol{x}, \boldsymbol{z})}{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\, d\boldsymbol{z}.
:::
Since the logarithm is concave, Jensen's inequality immediately yields a lower bound. This is the central theorem of the chapter.

```{prf:theorem} Evidence lower bound
:label: thm-elbo

For any density $q_\phi(\boldsymbol{z} | \boldsymbol{x})$ whose support contains the support of $p_\theta(\boldsymbol{z} | \boldsymbol{x})$,
:::{math}
\log p_\theta(\boldsymbol{x})
\geq
\mathbb{E}_{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\left[
\log p_\theta(\boldsymbol{x}, \boldsymbol{z})
-
\log q_\phi(\boldsymbol{z} | \boldsymbol{x})
\right].
:::
The right-hand side is called the evidence lower bound, or ELBO.
```

```{prf:proof}
Write
:::{math}
\log p_\theta(\boldsymbol{x})
=
\log \mathbb{E}_{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\left[
\frac{p_\theta(\boldsymbol{x}, \boldsymbol{z})}{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\right].
:::
Jensen's inequality for the concave logarithm implies
:::{math}
\log p_\theta(\boldsymbol{x})
\geq
\mathbb{E}_{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\left[
\log \frac{p_\theta(\boldsymbol{x}, \boldsymbol{z})}{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
\right].
:::
Expanding the logarithm gives the stated formula.
```

The **ELBO** is more than a technical inequality. It is the training objective that makes the model practical. Expanding the joint density yields
:::{math}
\mathcal{L}_{ELBO}(\theta, \phi; \boldsymbol{x})
=
\mathbb{E}_{q_\phi(\boldsymbol{z} | \boldsymbol{x})}
[\log p_\theta(\boldsymbol{x} | \boldsymbol{z})]
-
\operatorname{KL}(q_\phi(\boldsymbol{z} | \boldsymbol{x}) \| p(\boldsymbol{z})).
:::
The first term is a reconstruction term: it encourages the decoder to explain the observed image well from latent codes sampled from the encoder. The second term regularizes the approximate posterior toward the prior. This is what turns a deterministic autoencoder into a generative model. If the latent codes produced by the encoder remain close to a simple reference distribution, then sampling from that reference distribution at generation time becomes meaningful.

```{prf:theorem} ELBO decomposition with posterior gap
:label: thm-elbo-gap

For every observation $\boldsymbol{x}$,
:::{math}
\log p_\theta(\boldsymbol{x})
=
\mathcal{L}_{ELBO}(\theta, \phi; \boldsymbol{x})
+
\operatorname{KL}\big(q_\phi(\boldsymbol{z} | \boldsymbol{x}) \| p_\theta(\boldsymbol{z} | \boldsymbol{x})\big).
:::
```

```{prf:proof}
Starting from the ELBO expression,
:::{math}
\mathcal{L}_{ELBO}
=
\mathbb{E}_{q_\phi}
[\log p_\theta(\boldsymbol{x}, \boldsymbol{z}) - \log q_\phi(\boldsymbol{z} | \boldsymbol{x})].
:::
Add and subtract $\log p_\theta(\boldsymbol{z} | \boldsymbol{x})$. Using
:::{math}
\log p_\theta(\boldsymbol{x}, \boldsymbol{z})
=
\log p_\theta(\boldsymbol{z} | \boldsymbol{x}) + \log p_\theta(\boldsymbol{x}),
:::
we obtain
:::{math}
\mathcal{L}_{ELBO}
=
\log p_\theta(\boldsymbol{x})
-
\mathbb{E}_{q_\phi}
\left[
\log \frac{q_\phi(\boldsymbol{z} | \boldsymbol{x})}{p_\theta(\boldsymbol{z} | \boldsymbol{x})}
\right].
:::
The expectation is exactly the Kullback-Leibler divergence between the approximate and exact posterior, which proves the identity.
```

This identity explains why the bound is useful. Maximizing the ELBO simultaneously pushes upward the data log-likelihood and pushes downward the gap between the recognition model and the true posterior. The quality of inference and the quality of generation are therefore coupled. If the approximate posterior family is too restrictive, the model may learn a decoder that is easier for the encoder to support rather than one that is genuinely optimal for the data.

```{admonition} Numerical Example: Reading the ELBO Terms
:class: numerical-example

Suppose an encoder sees an image of a sneaker and outputs a two-dimensional Gaussian posterior with mean $\boldsymbol{\mu} = (1.2, -0.4)$ and standard deviations $(0.5, 0.8)$. The corresponding variational distribution says that the image is encoded near the point $(1.2, -0.4)$ in latent space, but with noticeable uncertainty, especially in the second coordinate.

If the decoder reconstructs the sneaker well, the reconstruction term in the ELBO is favorable. But the KL term also checks how far this posterior has drifted from the standard Gaussian prior. Here the first coordinate is shifted away from zero, so there is a regularization cost. The ELBO is therefore balancing two pressures at once: keep enough latent information to reconstruct the sneaker, but do not let the posterior wander arbitrarily far from the simple prior that we want to sample from at generation time.
```

There is also an important optimization lesson hidden here. The encoder $q_\phi(\boldsymbol{z} | \boldsymbol{x})$ is often called an amortized inference model because a single neural network is trained to solve approximate inference for every observation in the dataset at once. This is computationally powerful, but it also creates an additional approximation layer beyond the choice of variational family itself. The encoder may fail to represent the true posterior not only because the family is too simple, but also because the shared inference network does not allocate enough capacity to every observation equally well.

For a PhD audience, this is worth stressing because many later improvements to VAEs can be read as interventions on this exact bottleneck. Some methods enrich the variational family, some redesign the decoder likelihood, some modify the objective weighting, and some attack amortization error directly. The classical ELBO is therefore not the end of the story. It is the organizing baseline from which a whole research program begins.

## Gaussian Encoders and the Reparameterization Trick

In practice, a common choice is
:::{math}
q_\phi(\boldsymbol{z} | \boldsymbol{x}) =
\mathcal{N}(\boldsymbol{z}; \boldsymbol{\mu}_\phi(\boldsymbol{x}), \operatorname{diag}(\boldsymbol{\sigma}_\phi^2(\boldsymbol{x}))).
:::
The encoder network outputs the mean vector and the log-variance vector. If the prior is standard Gaussian, the KL term can be computed in closed form. The remaining challenge is differentiating through a stochastic sample from $q_\phi(\boldsymbol{z} | \boldsymbol{x})$. The reparameterization trick solves this by writing
:::{math}
\boldsymbol{z}
=
\boldsymbol{\mu}_\phi(\boldsymbol{x})
+
\boldsymbol{\sigma}_\phi(\boldsymbol{x}) \odot \boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}).
:::
Now the randomness is isolated in $\boldsymbol{\varepsilon}$, which does not depend on $\phi$. The map from parameters to sample becomes differentiable, and gradient-based learning becomes feasible.

## Guided Implementation

The code is placed here on purpose, immediately after the probabilistic ingredients are introduced. At this stage the reader should be able to ask very concrete questions: where is $q_\phi(\boldsymbol{z} | \boldsymbol{x})$ represented, where are $\boldsymbol{\mu}_\phi(\boldsymbol{x})$ and $\log \boldsymbol{\sigma}_\phi^2(\boldsymbol{x})$, where does the decoder define $p_\theta(\boldsymbol{x} | \boldsymbol{z})$, and where is the ELBO decomposition turned into an actual loss?

We use `FashionMNIST` because it is small enough for a course notebook but still diverse enough to make the trade-offs visible. Even on this toy-scale dataset one can already see the key phenomena: blurry reconstructions, latent regularization, posterior collapse, and the effect of changing the KL pressure.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchvision import datasets, transforms, utils
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
if device.type == "cuda":
    torch.cuda.manual_seed_all(7)
num_workers = 2 if device.type == "cuda" else 0
project_root = Path.cwd() if (Path.cwd() / "_config.yml").exists() else Path.cwd().parent
DATA_ROOT = project_root / "data"

# Settings
batch_size = 128
latent_dim = 32
base_channels = 32
lr = 2e-4
epochs = 60

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)


The hyperparameters deserve more interpretation than they usually receive in compact tutorials. The latent dimension is set to `32`. That is not sacred, but it expresses a modeling belief: the data should admit a representation that is **much lower-dimensional than pixel space**, while still rich enough to encode broad semantic variation. This is precisely where the **manifold hypothesis** becomes operational. The latent code is not literally the true manifold coordinate system, but it is a learned coordinate system that tries to track lower-dimensional explanatory structure.

If the latent dimension is too small, the model is forced into destructive compression and reconstructions degrade sharply. If it is too large, the encoder can hide too much information in the latent code and the regularizing role of the prior weakens. So the latent dimension is not just a capacity number. It is a statement about how compressible we believe the dataset is.

The training horizon is also intentionally longer than a tiny demo. A vanilla VAE on `FashionMNIST` often still looks undertrained after twenty or thirty epochs, especially if we want meaningful interpolation and metric comparisons. `60` epochs is still modest, but it is long enough that the qualitative behavior stops feeling like a throwaway toy run.


In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=32, base_channels=32):
        super().__init__()
        # 28x28 -> 14x14 -> 7x7.
        self.encoder = nn.Sequential(
            nn.Conv2d(1, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.SiLU(),
            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.SiLU(),
            nn.Flatten(),
        )
        encoded_dim = base_channels * 4 * 7 * 7
        self.mu_head = nn.Linear(encoded_dim, latent_dim)
        self.logvar_head = nn.Linear(encoded_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, encoded_dim),
            nn.SiLU(),
            nn.Unflatten(1, (base_channels * 4, 7, 7)),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.SiLU(),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, 1, kernel_size=3, padding=1),
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.mu_head(h)
        logvar = self.logvar_head(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        # Sample through parameter-free noise so gradients can flow to mu/logvar.
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        logits = self.decoder(z)
        return logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


model = VAE(latent_dim=latent_dim, base_channels=base_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

The method `reparameterize` is the code-level manifestation of the identity
:::{math}
\boldsymbol{z} = \boldsymbol{\mu}_\phi(\boldsymbol{x}) + \boldsymbol{\sigma}_\phi(\boldsymbol{x}) \odot \boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}).
:::
It is worth pausing here pedagogically. Without this rewrite, a direct sample from a parameterized Gaussian would obstruct backpropagation through the stochastic node. After the rewrite, the random source is independent of the learnable parameters, so the computational graph remains differentiable with respect to $\phi$.

## ELBO Loss in Code

We now implement the negative ELBO. The reconstruction term is approximated by a binary cross-entropy between the reconstructed image probabilities and the input image. This corresponds to choosing a Bernoulli observation model for pixels. The KL term is the closed-form divergence between a diagonal Gaussian posterior and the standard Gaussian prior. Since optimizers minimize rather than maximize, we return the negative ELBO.

In [ ]:
def elbo_loss(x, logits, mu, logvar):
    # Bernoulli reconstruction term for pixels in [0, 1].
    reconstruction = F.binary_cross_entropy_with_logits(
        logits,
        x,
        reduction="sum",
    )
    # Closed-form KL for a diagonal Gaussian encoder against N(0, I).
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    loss = reconstruction + kl
    return loss, reconstruction, kl

A subtle but important implementation detail is the use of logits instead of post-sigmoid probabilities inside `binary_cross_entropy_with_logits`. This is numerically more stable than applying a sigmoid manually and then feeding the result to a separate binary cross-entropy function. After training, however, we can still apply a sigmoid for visualization because we want actual pixel intensities in $[0,1]$.

The KL line in the function is equally important to decode for students. It is not a heuristic penalty chosen because Gaussian latents "feel natural". It is exactly the closed-form expression derived in Theorem {prf:ref}`thm-diagonal-gaussian-kl`, evaluated coordinatewise and summed across the minibatch. One of the nicest moments in teaching VAEs is when students realize that this compact piece of PyTorch code is not an approximation trick layered on top of theory, but the direct computational form of a theorem from the previous notebook.

A second important design choice is the reconstruction law. In this notebook the decoder outputs logits and we optimize a Bernoulli-style binary cross-entropy. For grayscale images scaled to $[0,1]$, this is a reasonable classroom choice. But it is not the only possible observation model, and it is part of why simple VAEs often look softer than GANs or diffusion models. Pixelwise likelihoods reward the model for being locally correct, not necessarily perceptually sharp.

## Training Dynamics and the Meaning of the KL Weight

Even when one writes down the vanilla ELBO, the effective weight of the KL term is one of the most important practical quantities in the model. If the KL pressure is too strong too early, the encoder may learn to communicate too little and the latent variable becomes uninformative. If the KL pressure is too weak, reconstructions may look excellent while samples from the prior become poor because the encoded latent space drifts too far from the distribution used at generation time.

This is why the split between reconstruction and KL should be monitored during training. Those two numbers are not decorative logging. They tell us whether the latent channel is being used, whether regularization is dominating, and whether the model has found a healthy compromise.

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_reconstruction = 0.0
    total_kl = 0.0

    for x, _ in tqdm(loader, desc="train", leave=False):
        x = x.to(device)

        optimizer.zero_grad()
        logits, mu, logvar = model(x)
        loss, reconstruction, kl = elbo_loss(x, logits, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_reconstruction += reconstruction.item()
        total_kl += kl.item()

    n = len(loader.dataset)
    return {
        "loss": total_loss / n,
        "reconstruction": total_reconstruction / n,
        "kl": total_kl / n,
    }


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_reconstruction = 0.0
    total_kl = 0.0

    for x, _ in tqdm(loader, desc="eval", leave=False):
        x = x.to(device)
        logits, mu, logvar = model(x)
        loss, reconstruction, kl = elbo_loss(x, logits, mu, logvar)

        total_loss += loss.item()
        total_reconstruction += reconstruction.item()
        total_kl += kl.item()

    n = len(loader.dataset)
    return {
        "loss": total_loss / n,
        "reconstruction": total_reconstruction / n,
        "kl": total_kl / n,
    }

In [ ]:
history = {"train_loss": [], "train_kl": [], "val_loss": [], "val_kl": []}

for epoch in tqdm(range(epochs), desc="VAE epochs"):
    train_stats = train_epoch(model, train_loader, optimizer, device)
    val_stats = evaluate(model, test_loader, device)

    history["train_loss"].append(train_stats["loss"])
    history["train_kl"].append(train_stats["kl"])
    history["val_loss"].append(val_stats["loss"])
    history["val_kl"].append(val_stats["kl"])

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train loss: {train_stats['loss']:.4f} | "
        f"train KL: {train_stats['kl']:.4f} | "
        f"val loss: {val_stats['loss']:.4f} | "
        f"val KL: {val_stats['kl']:.4f}"
    )

A harmonious VAE run on `FashionMNIST` usually has a recognizable shape. The reconstruction term falls steadily. The KL term rises away from zero and then stabilizes instead of collapsing. Reconstructions become recognizable early, but prior samples improve more slowly because they depend on the latent space aligning with the prior rather than only on per-example encoding quality.

This is one reason VAEs remain such good teaching models: they let students inspect *why* a training run is working or failing instead of only reporting one opaque objective value.

## Reconstructions, Samples, and Latent Geometry

A VAE should be read through several lenses at once. Reconstructions tell us whether the encoder-decoder pair is transmitting enough information. Prior samples tell us whether the learned latent space remains compatible with the prior we want to sample from. Interpolations tell us whether the latent space has acquired a reasonably smooth semantic geometry rather than becoming a disconnected codebook.

In [ ]:
@torch.no_grad()
def show_reconstructions(model, loader, device, n=8):
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    logits, _, _ = model(x)
    # Sigmoid is only for visualization; training used logits directly.
    recon = torch.sigmoid(logits).view(-1, 1, 28, 28)

    grid = torch.cat([x.cpu(), recon.cpu()], dim=0)
    image = utils.make_grid(grid, nrow=n, pad_value=1.0)
    plt.figure(figsize=(1.5 * n, 3.0))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()


@torch.no_grad()
def show_samples(model, device, n=16):
    model.eval()
    z = torch.randn(n, latent_dim, device=device)
    logits = model.decode(z)
    samples = torch.sigmoid(logits).view(-1, 1, 28, 28).cpu()
    image = utils.make_grid(samples, nrow=4, pad_value=1.0)
    plt.figure(figsize=(6, 6))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()

In [ ]:
@torch.no_grad()
def interpolate(model, loader, device, steps=8):
    model.eval()
    x, _ = next(iter(loader))
    x0 = x[0:1].to(device)
    x1 = x[1:2].to(device)

    # Interpolate between posterior means to visualize latent geometry.
    mu0, _ = model.encode(x0)
    mu1, _ = model.encode(x1)

    alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1)
    z = (1 - alphas) * mu0 + alphas * mu1
    logits = model.decode(z)
    images = torch.sigmoid(logits).view(-1, 1, 28, 28).cpu()

    grid = utils.make_grid(images, nrow=steps, pad_value=1.0)
    plt.figure(figsize=(1.7 * steps, 2.5))
    plt.imshow(grid.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()


interpolate(model, test_loader, device)

Interpolation is particularly revealing when discussing the relation between VAEs and the manifold hypothesis. If linear motion in latent space produces gradual semantic motion in image space, that is evidence that the model has learned a coordinate system with some geometric coherence. It does not prove that the latent space is the true data manifold, but it does show that the model is organizing variation in a more structured way than raw pixel space does.

## Quantitative Evaluation

Qualitative inspection is essential, but it is not enough. We can also compare the generated distribution against real images through **FID** and **KID**. On a small grayscale dataset these numbers should be treated as teaching diagnostics rather than benchmark claims, but they are still useful because they make distribution-level evaluation concrete.

In [ ]:
def prepare_for_inception_metrics(images):
    # The default Inception feature extractor expects RGB-like inputs.
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)
    return images.clamp(0.0, 1.0)


@torch.no_grad()
def compute_vae_fid_and_kid(model, real_loader, device, num_fake=1000, metric_batch_size=64):
    fid = FrechetInceptionDistance(
        feature=2048,
        normalize=True,
        reset_real_features=False,
    ).set_dtype(torch.float64).to(device)
    kid = KernelInceptionDistance(
        feature=2048,
        subsets=10,
        subset_size=100,
        normalize=True,
        reset_real_features=False,
    ).to(device)

    # First accumulate features from real images in small batches.
    seen_real = 0
    real_target = num_fake
    for real_images, _ in tqdm(real_loader, desc="VAE real metrics", leave=False):
        remaining = real_target - seen_real
        if remaining <= 0:
            break
        real_images = real_images[: min(metric_batch_size, remaining)].to(device)
        real_images = prepare_for_inception_metrics(real_images)
        fid.update(real_images, real=True)
        kid.update(real_images, real=True)
        seen_real += real_images.size(0)

    # Then accumulate features from generated samples.
    generated = 0
    pbar = tqdm(total=num_fake, desc="VAE fake metrics", leave=False)
    while generated < num_fake:
        batch_n = min(metric_batch_size, num_fake - generated)
        z = torch.randn(batch_n, latent_dim, device=device)
        logits = model.decode(z)
        fake_images = torch.sigmoid(logits).view(-1, 1, 28, 28)
        fake_images = prepare_for_inception_metrics(fake_images)
        fid.update(fake_images, real=False)
        kid.update(fake_images, real=False)
        generated += batch_n
        pbar.update(batch_n)
    pbar.close()

    kid_mean, kid_std = kid.compute()
    return {
        "fid": fid.compute().item(),
        "kid_mean": kid_mean.item(),
        "kid_std": kid_std.item(),
    }


metric_scores = compute_vae_fid_and_kid(model, test_loader, device)
print(metric_scores)

This evaluation block also clarifies one of the VAE's characteristic tradeoffs. A VAE may reconstruct cleanly and organize latent space nicely, yet still obtain weaker FID or KID than a sharper generator because the decoded samples can look slightly smooth or blurry in feature space. That does not make the model useless. It means the metric is emphasizing a specific aspect of generative quality, namely closeness of the generated feature distribution to the real one.

```{prf:theorem} Closed-form KL for a diagonal Gaussian encoder
:label: thm-diagonal-gaussian-kl

Let
:::{math}
q_\phi(\boldsymbol{z} | \boldsymbol{x})
=
\mathcal{N}\big(
    \boldsymbol{z};
    \boldsymbol{\mu},
    \operatorname{diag}(\boldsymbol{\sigma}^2)
\big)
:::
and let the prior be
:::{math}
p(\boldsymbol{z}) = \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}).
:::
Then
:::{math}
\operatorname{KL}(q_\phi(\boldsymbol{z} | \boldsymbol{x}) \| p(\boldsymbol{z}))
=
\frac{1}{2}
\sum_{j=1}^d
\left(
    \mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1
\right).
:::
```

```{prf:proof}
By definition,
:::{math}
\operatorname{KL}(q \| p)
=
\mathbb{E}_q[\log q(\boldsymbol{z}) - \log p(\boldsymbol{z})].
:::
For the diagonal Gaussian posterior,
:::{math}
\log q(\boldsymbol{z})
=
-\frac{1}{2}
\sum_{j=1}^d
\left[
    \log(2\pi \sigma_j^2)
    +
    \frac{(z_j-\mu_j)^2}{\sigma_j^2}
\right],
:::
while for the standard Gaussian prior,
:::{math}
\log p(\boldsymbol{z})
=
-\frac{1}{2}
\sum_{j=1}^d
\left[
    \log(2\pi)
    +
    z_j^2
\right].
:::
Subtracting and taking expectation under $q$, we use
:::{math}
\mathbb{E}_q[(z_j-\mu_j)^2] = \sigma_j^2,
\qquad
\mathbb{E}_q[z_j^2] = \mu_j^2 + \sigma_j^2.
:::
The constant $\log(2\pi)$ terms cancel, leaving
:::{math}
\operatorname{KL}(q \| p)
=
\frac{1}{2}
\sum_{j=1}^d
\left(
    \mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1
\right),
:::
which proves the formula.
```

This explicit KL formula is one of the practical reasons the Gaussian VAE became such a standard teaching example. The reconstruction term must still be estimated through stochastic samples, but the regularization term is analytically available. The resulting objective is therefore simple enough to implement directly while still expressing the real variational logic of the model.

The broader lesson is that a VAE is best understood as a *coupled system*: probabilistic assumptions, latent geometry, decoder likelihood, architecture, and optimization schedule all interact. That is why a chapter on VAEs should feel more like a continuous argument than a proof section followed by an unrelated coding appendix.

The VAE should be remembered as the canonical bridge between probabilistic latent-variable modeling and neural network training. Its historical formulations were introduced in {cite}`kingma2013auto` and {cite}`rezende2014stochastic`. Many later models, including latent diffusion systems, become easier to interpret once one has internalized the VAE habit of thought: choose a latent construction, derive a tractable surrogate objective, and let neural networks amortize the hard parts.